In [7]:
from pathlib import Path
import xarray as xr
import sys

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [8]:
Processed_Dataset = "20261001_pallas_gaussian"

In [9]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [10]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [11]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 29682, range: 383)> Size: 91MB
array([[0.35888847, 0.35264826, 0.34524981, ...,        nan,        nan,
               nan],
       [0.35883353, 0.35259485, 0.34519792, ...,        nan,        nan,
               nan],
       [0.35877883, 0.35254169, 0.34514626, ...,        nan,        nan,
               nan],
       ...,
       [0.41341745, 0.40798716, 0.4013619 , ...,        nan,        nan,
               nan],
       [0.41341745, 0.40798716, 0.4013619 , ...,        nan,        nan,
               nan],
       [0.41341745, 0.40798716, 0.4013619 , ...,        nan,        nan,
               nan]], shape=(29682, 383))
Coordinates:
  * Time     (Time) datetime64[ns] 237kB 2026-10-01T00:43:14.052958488 ... 20...
  * range    (range) float64 3kB 216.0 252.0 288.0 ... 1.094e+04 1.097e+04
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    model:      ITU-R P.676
<xarray.DataArray 'alpha_35GHz' (Time: 29682, range:

In [12]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")